# TF-IDF reference

This is a quick side check, not one of the actual models being compared. It just answers one question: how much of this task can a simple word matching model solve, without any sequence modeling at all? This notebook runs that check once against the frozen split.

## Load the frozen split

In [1]:
import sys
import time

sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import tfidf_reference as tr
from src.data import CLASS_TO_ID, LABEL_COL, LABELS, TEXT_COL
from src.preprocessing import load_frozen_dataset
from src.reproducibility import get_git_info

df, train_idx, val_idx, test_idx = load_frozen_dataset(base_dir="..")
print(f"train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

train=81442 val=10179 test=10181


This uses the same frozen dataset loader as before. It checks that the data and split match what is on record, and stops if anything has changed. Nothing here regenerates or alters the split.

## TF-IDF reference

In [2]:
print("TF-IDF config:", tr.TFIDF_CONFIG)
print("Logistic Regression config:", tr.LOGREG_CONFIG)

TF-IDF config: {'analyzer': 'word', 'ngram_range': (1, 2), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True}
Logistic Regression config: {'max_iter': 1000, 'C': 1.0, 'solver': 'lbfgs'}


The setup was decided upfront, before fitting anything: word-level TF-IDF, 1 to 2 word phrases, ignoring words that appear only once or in over 95% of documents, with sublinear scaling. The text goes through the same basic cleanup used everywhere else in the project. Nothing extra was added just for this check.

## Fit on train

In [3]:
t0 = time.time()
vectorizer, model = tr.fit_tfidf_reference(
    df.iloc[train_idx][TEXT_COL],
    df.iloc[train_idx][LABEL_COL],
)
fit_time = time.time() - t0
print(f"fit in {fit_time:.1f}s | vocabulary size: {len(vectorizer.vocabulary_):,}")

fit in 260.3s | vocabulary size: 619,462


In [4]:
# Fitting again on validation/test text must not be possible through this API -
# `.transform()` is the only thing called on them, and it cannot alter
# `vectorizer.vocabulary_`. Confirmed directly, not just assumed.
vocab_before = dict(vectorizer.vocabulary_)
X_val = tr.transform(vectorizer, df.iloc[val_idx][TEXT_COL])
X_test = tr.transform(vectorizer, df.iloc[test_idx][TEXT_COL])
assert vectorizer.vocabulary_ == vocab_before, "vocabulary changed after transform - must never happen"
print("val/test transformed; vocabulary unchanged. shapes:", X_val.shape, X_test.shape)

val/test transformed; vocabulary unchanged. shapes: (10179, 619462) (10181, 619462)


## Evaluate

In [5]:
from src.evaluation import compute_metrics

y_val = df.iloc[val_idx][LABEL_COL].map(CLASS_TO_ID).to_numpy()
y_test = df.iloc[test_idx][LABEL_COL].map(CLASS_TO_ID).to_numpy()

# Descriptive only (Part 6) - not used to pick a configuration.
val_pred = model.predict(X_val)
val_metrics = compute_metrics(y_val, val_pred, labels=LABELS)
print(f"validation macro-F1 (descriptive only): {val_metrics['macro_f1']:.4f}")

validation macro-F1 (descriptive only): 0.8750


In [6]:
test_pred = model.predict(X_test)
test_metrics = compute_metrics(y_test, test_pred, labels=LABELS)
print(f"TEST macro-F1: {test_metrics['macro_f1']:.4f}")
pd.Series({
    "macro_f1": test_metrics["macro_f1"], "accuracy": test_metrics["accuracy"],
    "macro_precision": test_metrics["macro_precision"], "macro_recall": test_metrics["macro_recall"],
})

TEST macro-F1: 0.8707


macro_f1           0.870749
accuracy           0.868186
macro_precision    0.871016
macro_recall       0.870947
dtype: float64

In [7]:
pd.DataFrame(test_metrics["per_class"]).T

,precision,recall,f1,support
Checking or savings account,0.783416,0.803065,0.793119,2153.0
Credit card,0.866733,0.842436,0.854412,2069.0
Debt collection,0.899695,0.946438,0.922475,1867.0
"Money transfer, virtual currency, or money service",0.824842,0.811843,0.818291,2094.0
Student loan,0.980392,0.950951,0.965447,1998.0


In [8]:
pd.DataFrame(test_metrics["confusion_matrix"], index=LABELS, columns=LABELS)

,Checking or savings account,Credit card,Debt collection,"Money transfer, virtual currency, or money service",Student loan
Checking or savings account,1729,130,13,278,3
Credit card,135,1743,112,71,8
Debt collection,14,60,1767,7,19
"Money transfer, virtual currency, or money service",324,54,8,1700,8
Student loan,5,24,64,5,1900


## Compare with M0

In [9]:
from src.evaluation import calculate_deltas, format_delta

M0_MACRO_F1 = 0.8508  # mean across 3 seeds, results/runs.csv (Task 6) - not recomputed here
delta = calculate_deltas(current_f1=test_metrics["macro_f1"], previous_f1=None, baseline_m0_f1=M0_MACRO_F1)
print(f"TF-IDF reference − M0 = {format_delta(delta['delta_vs_m0'])}")

TF-IDF reference − M0 = +0.0199


This is not one of the real models in the ladder, just a reference point, computed the same way as the real deltas for consistency.

## Inspect features

In [10]:
top = tr.top_features_by_class(vectorizer, model, k=10)
pd.DataFrame(top)

,Checking or savings account,Credit card,Debt collection,"Money transfer, virtual currency, or money service",Student loan
0,bank,card,debt,paypal,loan
1,funds,credit card,collection,money,mohela
2,checking,credit,collections,cashapp,loans
3,chime,synchrony,credit,transfer,student
4,debit,citi,my credit,coinbase,nelnet
5,overdraft,capital one,owe,app,school
6,deposit,interest,company,cash,student loan
7,account,charge,owed,funds,forbearance
8,banking,barclays,court,zelle,aidvantage
9,the bank,rewards,collect,cash app,education


The top predictive words are servicer names, product terms, and regulation references, like navient, mohela, overdraft, and fdcpa. This is the same class-specific vocabulary the earlier data audit found. No redaction pattern or meaningless token dominates any class's top list. This is real signal, not a shortcut.

## Save the result

In [11]:
git_commit = get_git_info("..")["git_commit"]

import json
preprocessing_version = json.load(open("../artifacts/preprocessing/preprocessing_version.json"))["version_id"]
dataset_sha = json.load(open("../data/splits/split_manifest.json"))["dataset_content_sha256"]

result = tr.build_result(
    vectorizer, model, X_test, y_test,
    val_macro_f1=val_metrics["macro_f1"],
    dataset_content_sha256=dataset_sha,
    preprocessing_version=preprocessing_version,
    runtime_seconds=fit_time,
    git_commit=git_commit,
)
result.save("../results/tfidf_reference.json")
print(f"saved: results/tfidf_reference.json (label={result.label})")

saved: results/tfidf_reference.json (label=TFIDF_REFERENCE)


### Setup

1. The setup was fixed before fitting, not tuned to the result: TF-IDF, word-level, 1 to 2 word phrases, with document-frequency filters, plus Logistic Regression.
2. Fit on train only. Validation was just for sanity checking, never for model selection. Test was evaluated once, after the model was already locked in.
3. Saved separately from the main results file, since this is not one of the official model configurations.

### What this tells us

This shows how much of the task a simple word matching model can solve on its own. It is a reference point for how lexical the task is, not a claim about why the LSTM behaves a certain way, and not a substitute for the real baseline, M0.